In [ ]:
import os
from datasets import load_dataset  

meu_dataset = load_dataset('json', data_files={
    'train': '/home/cecilia/Documentos/PIBIC/Fase1/dados_svm_bert/train_bio_processado.json',
    'validation': '/home/cecilia/Documentos/PIBIC/Fase1/dados_svm_bert/val_bio_processado.json',
    'test': '/home/cecilia/Documentos/PIBIC/Fase1/dados_svm_bert/test_bio_processado.json'
})



In [ ]:

label_to_id = {"O": 0, "B-ASP": 1, "I-ASP": 2} 

In [4]:
label_list = ["O", "B-ASP", "I-ASP"]

In [ ]:
from transformers import AutoModelForTokenClassification
from transformers import AutoTokenizer



model_name = "google-bert/bert-base-multilingual-uncased" 

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3
)

In [ ]:

def tokenize_and_align_labels(examples): 
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding="max_length"
    )

    labels = []
    for i, label in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_to_id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:

colunas_para_retirar = meu_dataset['train'].column_names 



In [ ]:
tokenized_datasets = meu_dataset.map( 
    batched=True,
    remove_columns=colunas_para_retirar)

In [ ]:
from transformers import DataCollatorForTokenClassification


data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
pip install seqeval

In [11]:
from seqeval.metrics import f1_score, precision_score, recall_score
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "Precision": precision_score(true_labels, true_predictions),
        "Recall": recall_score(true_labels, true_predictions),
        "F1-score": f1_score(true_labels, true_predictions),
    }

In [ ]:
from transformers import TrainingArguments, Trainer


args = TrainingArguments(
    output_dir="resultados",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    load_best_model_at_end=True,
)



trainer = Trainer(
    model=model,
    args=args,
    data_collator = data_collator,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics = compute_metrics
)

#trainer.train()

In [ ]:
trainer.train() 

In [ ]:

metricas_finais = trainer.evaluate(tokenized_datasets["test"])
print(metricas_finais)